# IND320 Project work 4

**Author:** Inge Gilje Gunnarshaug

## 1. Introduction

This is the notebook for the fourth assignment in IND320. The corresponding streamlit app and github repository can be found here:

### Project links

- **GitHub repository:** https://github.com/IngeGiljeGunnarshaug/ind320-inge-gilje-gunnarshaug
- **Streamlit app:** https://ind320-inge-gilje-gunnarshaug.streamlit.app/

## 2. Environment and Imports

In [1]:
import requests
import pandas as pd
from io import StringIO
import plotly.express as px

## 3. Euro and USD currency exchange rates
### 3.1 Download function

In [2]:
def download_exchange_rates(start_date, end_date):
    url = "https://data.norges-bank.no/api/data/EXR/B.EUR+USD.NOK.SP"

    params = {
        "startPeriod": start_date,
        "endPeriod": end_date,
        "format": "csvfile"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    df = pd.read_csv(StringIO(response.text))

    df = df[["TIME_PERIOD", "BASE_CUR", "OBS_VALUE"]]

    df = df.rename(columns={
        "TIME_PERIOD": "date",
        "BASE_CUR": "currency",
        "OBS_VALUE": "exchange_rate"
    })

    df["date"] = pd.to_datetime(df["date"])

    df = df.pivot(
        index="date",
        columns="currency",
        values="exchange_rate"
    )

    df = df.sort_index()
    df.columns.name = None

    return df

### 3.2 Verify that the function works

In [3]:
df_currency = download_exchange_rates(
    start_date="2020-01-01",
    end_date="2026-11-25"
)

display(df_currency.head())
display(df_currency.tail())

print("Number of observations:", len(df_currency))
print("\nMissing values:")
print(df_currency.isna().sum())

,EUR,USD
date,,
2020-01-02,9.8408,8.7919
2020-01-03,9.8315,8.8199
2020-01-06,9.8488,8.7983
2020-01-07,9.8548,8.8210
2020-01-08,9.8508,8.8626


,EUR,USD
date,,
2026-11-19,10.6475,9.6866
2026-11-20,10.7180,9.7525
2026-11-23,10.7285,9.8076
2026-11-24,10.7350,9.7920
2026-11-25,10.7778,9.8562


Number of observations: 1740

Missing values:
EUR    0
USD    0
dtype: int64


In [5]:
fig = px.line(
    df_currency,
    title="Historical EUR and USD Exchange Rates",
    labels={
        "date": "Date",
        "value": "Exchange rate (NOK)",
        "variable": "Currency"
    }
)

fig.show()